# Tutorial 70 — Retrieve by semantic similarity

[Tutorial 60](./60_embeddings.ipynb) showed how an embedding model represents
chunks and questions as vectors. This tutorial builds a reusable semantic index
and ranks repository chunks by similarity in embedding space.

Semantic retrieval is meaning-oriented: a useful chunk can rank highly even
when it does not repeat the question's exact words. You will build the index,
search it, interpret cosine-similarity scores, and compare the loop and NumPy
comparison implementations.

No chat model writes an answer yet. [Tutorial 80](./80_lexical_retrieval.ipynb)
adds a complementary retrieval channel based on exact wording and software
identifiers.


## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import qachat


## 2. Load an embedding model

Semantic retrieval uses the same embedding layer introduced in
[Tutorial 60](./60_embeddings.ipynb). QAChat resolves `auto` first and selects
the corresponding canonical embedding-model default. The chat model that
eventually generates answers is independent from this model.


In [ ]:
REQUESTED_BACKEND = qachat.BACKEND_AUTO

SELECTED_BACKEND = qachat.choose_backend(REQUESTED_BACKEND)

MODEL_ID = (
    qachat.DEFAULT_EMBEDDING_MODEL_MLX
    if SELECTED_BACKEND == "mlx"
    else qachat.DEFAULT_EMBEDDING_MODEL_TORCH
)

print(f"Requested backend: {REQUESTED_BACKEND}")
print(f"Resolved backend: {SELECTED_BACKEND}")
print(f"Embedding model: {MODEL_ID}")


In [ ]:
ALLOW_DOWNLOAD = True


In [ ]:
embedding_session = qachat.EmbeddingSession(
    MODEL_ID,
    backend=SELECTED_BACKEND,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


## 3. Prepare repository chunks

The following cells repeat the file-selection and chunking pipeline from
[Tutorial 40](./40_repository_files.ipynb) and
[Tutorial 50](./50_source_structure.ipynb) so this notebook remains
independently runnable. The embedding tokenizer supplies the chunk-size metric.


In [ ]:
report = qachat.scan_repository(
    root,
    extensions=[".py", ".md"],
    include_paths=["qachat", "README.md"],
)

print(f"Selected {len(report['selected_files'])} files:")

for i, file in enumerate(report["selected_files"]):
    print(f"{i}. {file}")


In [ ]:
source_files = qachat.read_repository_files(report)
units = qachat.extract_source_units(source_files)

print(f"Read {len(source_files)} files")
print(f"Natural units: {len(units)}")


In [ ]:
chunks = qachat.chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Searchable chunks: {len(chunks)}")


## 4. Build the semantic index

Indexing happens before a question is searched. Each repository chunk is
embedded once and stored with its vector and source metadata. Those document
embeddings can then be reused for many different queries.


In [ ]:
semantic_index = qachat.build_semantic_index(
    chunks,
    embedding_session,
)

assert len(semantic_index.chunks) == len(chunks)
assert len(semantic_index.embeddings) == len(chunks)

print(f"Indexed chunks: {len(semantic_index.chunks)}")
print(f"Vector size: {len(semantic_index.embeddings[0])}")


In [ ]:
first_chunk = semantic_index.chunks[0]
first_vector = semantic_index.embeddings[0]

print(f"Chunk ID: {first_chunk.identifier}")
print(f"Source: {first_chunk.path}")
print(first_chunk.text)
print(f"Stored vector: {first_vector}")


## 5. Search by meaning

The new question is formatted for the embedding model and embedded with the same
model used for the repository chunks. `semantic_search()` compares that query
vector with the stored document vectors and ranks the most similar chunks first.

The question intentionally avoids an exact function name so the example tests
meaning-oriented retrieval rather than a trivial identifier match.


In [ ]:
QUESTION = "How does QAChat decide which computation backend to run on?"

query_text = qachat.format_qwen3_query(
    instruction=qachat.SOFTWARE_REPOSITORY_INSTRUCTION,
    question=QUESTION,
)

print(query_text)


In [ ]:
results = qachat.semantic_search(
    semantic_index,
    query_text,
    embedding_session,
    count=5,
    backend="loop",
)

results


In [ ]:
scores = [result.score for result in results]

assert scores == sorted(scores, reverse=True)

print(f"Retrieved chunks: {len(results)}")
print(f"Highest score: {scores[0]}")


## 6. Inspect the semantic ranking

Cosine similarity determines ordering within this semantic channel. The score
is useful for ranking candidates under the same representation, but it is not a
probability that a chunk is correct and should not be compared directly with
scores produced by a different retrieval method.


In [ ]:
for rank, result in enumerate(results, start=1):
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    name = chunk.symbol or chunk.section or "(module context)"
    print(f"{rank}. {result.score:.3f} | {location} | {name}")


In [ ]:
best = results[0]

print("Most relevant chunk:\n")
print(best.chunk.text)


In [ ]:
for rank, result in enumerate(results, start=1):
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    name = chunk.symbol or chunk.section or "(module context)"
    print(
        f"\n>>> Chunk {rank}. {result.score:.3f} | {location} | {name} <<<\n"
    )
    print(chunk.text)


## 7. Choose the vector-comparison method

MLX or PyTorch runs the embedding model. The later `backend="loop"` and
`backend="numpy"` arguments control only how an already-computed query vector is
compared with stored vectors.

The loop implementation is simple for small indexes. NumPy can compare a larger
matrix efficiently; when the same index serves many questions, the matrix can be
built once and reused. Both methods should preserve the same ranking semantics.


### Temporary NumPy matrix

Setting `backend="numpy"` selects NumPy. If no matrix is supplied,
the function builds one for this search and then discards it.

In [ ]:
temporary_numpy_results = qachat.semantic_search(
    semantic_index,
    query_text,
    embedding_session,
    count=5,
    backend="numpy",
)

print(f"Retrieved chunks: {len(temporary_numpy_results)}")


### Reusable NumPy matrix

For repeated searches, build the matrix once and pass the same object to
every search. Rebuild it only when the semantic index changes.

In [ ]:
matrix = qachat.semantic_matrix(semantic_index)

print(f"Matrix shape: {matrix.shape}")
print(f"Matrix size: {matrix.nbytes / 1_000_000:.2f} MB")


In [ ]:
reusable_numpy_results = qachat.semantic_search(
    semantic_index,
    query_text,
    embedding_session,
    count=5,
    backend="numpy",
    matrix=matrix,
)

loop_ids = [result.chunk.identifier for result in results]
numpy_ids = [result.chunk.identifier for result in reusable_numpy_results]

assert loop_ids == numpy_ids

print(f"Loop ranking: {loop_ids}")
print(f"NumPy ranking: {numpy_ids}")


### Relation to Project Chat

The lower-level `semantic_search()` function exposes both the loop and NumPy
comparison methods so they can be tested directly here. The current
`RAGSession` API uses the default semantic-search path internally and does not
expose a per-question `semantic_method` argument.

This distinction is intentional in the tutorial: the section above demonstrates
an implementation choice inside semantic retrieval without implying that the
choice is part of the current Project Chat user API.


## 8. Release the model


In [ ]:
embedding_session.close()

print("Embedding model released.")


## Continue the tutorial series

**Previous:** [Tutorial 60 — Create embeddings](./60_embeddings.ipynb)

**Next:** [Tutorial 80 — Retrieve by lexical similarity](./80_lexical_retrieval.ipynb)
